In [6]:
LAST_NAME = "ELIAS"
SEED_NUM = 7
FAVORITE_ARTIST = "TUBERO"

execution_log = []

def monitor(func):
    def wrapper(*args, **kwargs):
        execution_log.append(f"{func.__name__} executed")
        return func(*args, **kwargs)
    return wrapper

@monitor
def telemetry_stream(last, seed, artist):
    base = [ord(c) for c in last[:3] + artist[:3]]
    for val in base:
        yield (val + seed) % 250  # keep values bounded

@monitor
def process_stream(stream):
    readings, valid, invalid, abnormal = [], 0, 0, []
    # Lambda to normalize values
    def normalize(x): return x if x >= 0 else 0
    for val in stream:
        try:
            nval = normalize(val)
            readings.append(nval)
            if 50 <= nval <= 180:
                valid += 1
            else:
                invalid += 1
                abnormal.append(nval)
        except Exception:
            invalid += 1
    return readings, valid, invalid, abnormal

@monitor
def trace_abnormal(val, calls=0, seq=None):
    if seq is None:
        seq = []
    seq.append(val)
    calls += 1
    if val < 50:  # base condition
        return seq, calls
    return trace_abnormal(val // 2, calls, seq)

stream = telemetry_stream(LAST_NAME, SEED_NUM, FAVORITE_ARTIST)
readings, valid, invalid, abnormal = process_stream(stream)

recursive_seq, recursive_calls = ([], 0)
if abnormal:
    recursive_seq, recursive_calls = trace_abnormal(abnormal[0])
    execution_log.append("trace_abnormal executed")

print("Student-Specific Inputs:", LAST_NAME, SEED_NUM, FAVORITE_ARTIST)
print("Generated Telemetry Data:", readings)
print("Valid/Invalid Results:", f"Valid={valid}, Invalid={invalid}")
print("Processed Results:", f"Abnormal={abnormal}")
print("Recursive Analysis:", f"Trace={recursive_seq}, Calls={recursive_calls}")
print("Final Diagnostic Summary:",
      f"Processed={len(readings)}, Valid={valid}, Invalid={invalid}, Abnormal={len(abnormal)}")
print("Execution Log:", execution_log)
print("Final Output:", "Status=Optimal" if valid > invalid else "Status=Faulty")

Student-Specific Inputs: ELIAS 7 TUBERO
Generated Telemetry Data: [76, 83, 80, 91, 92, 73]
Valid/Invalid Results: Valid=6, Invalid=0
Processed Results: Abnormal=[]
Recursive Analysis: Trace=[], Calls=0
Final Diagnostic Summary: Processed=6, Valid=6, Invalid=0, Abnormal=0
Execution Log: ['telemetry_stream executed', 'process_stream executed']
Final Output: Status=Optimal
